In [ ]:
import pandas as pd

df = pd.read_excel(
    "dukes_raw/DUKES_5.7_Plant capacity, United Kingdom.xlsx",
    sheet_name="5.7",
    header=None
)

print("=== DUKES 5.7 first 20 rows (raw content) ===")
for i in range(min(25, len(df))):
    print(f"Row {i}:", df.iloc[i].tolist())

print("\n=== Table shape ===")
print(f"Total rows: {len(df)}, Total columns: {len(df.columns)}")

In [ ]:
import pandas as pd
import os

FILE = "dukes_raw/DUKES_5.7_Plant capacity, United Kingdom.xlsx"
SHEET = "5.7"

df_raw = pd.read_excel(FILE, sheet_name=SHEET, header=None)

header_row_idx = 6
first_table_end = df_raw.index[
    df_raw.iloc[:, 0].astype(str).str.contains("Table 5.7.B", na=False)
].min()

cols = df_raw.iloc[header_row_idx].tolist()

df = df_raw.iloc[header_row_idx + 1:first_table_end].copy()
df.columns = cols

df = df.rename(columns={
    cols[0]: "generator_type",
    cols[1]: "fuel"
})

df = df[
    df["generator_type"].astype(str).str.strip().eq("All generating companies")
].copy()

df = df[
    ~df["fuel"].astype(str).str.contains("Total", case=False, na=False)
].copy()

rename_year_cols = {}
year_cols = []

for col in df.columns:
    clean = str(col).split("[")[0].replace(".0", "").strip()
    if clean.isdigit():
        y = int(clean)
        rename_year_cols[col] = y
        if 2010 <= y <= 2024:
            year_cols.append(y)

df = df.rename(columns=rename_year_cols)

fuel_to_tech = {
    "Coal fired": "coal",
    "Oil fired": "other_thermal",
    "Gas fired": "gas",
    "Mixed or dual fuelled": "other_thermal",
    "Nuclear stations": "nuclear",
    "Hydro (natural flow)": "hydro",
    "Pumped hydro": "storage",
    "Onshore wind": "wind_total",
    "Offshore wind": "wind_total",
    "Wave and tidal": "other_renewable",
    "Solar": "solar",
    "Bioenergy and waste": "biomass",
    "Other fossil fuels": "other_thermal",
}

df_long = df.melt(
    id_vars=["fuel"],
    value_vars=year_cols,
    var_name="year",
    value_name="capacity_mw"
)

df_long["year"] = df_long["year"].astype(int)
df_long["capacity_mw"] = pd.to_numeric(df_long["capacity_mw"], errors="coerce")
df_long["fuel"] = df_long["fuel"].astype(str).str.strip()
df_long["tech"] = df_long["fuel"].map(fuel_to_tech)

unmapped = sorted(df_long.loc[df_long["tech"].isna(), "fuel"].dropna().unique())
assert not unmapped, f"Unmapped DUKES 5.7 fuel labels: {unmapped}"

df_long = df_long.dropna(subset=["capacity_mw", "tech"]).copy()

df_final = (
    df_long
    .groupby(["year", "tech"], as_index=False)
    .agg(
        capacity_mw=("capacity_mw", "sum"),
        source_table=("fuel", lambda s: "DUKES 5.7: " + "; ".join(sorted(set(s))))
    )
)

df_final = df_final[["year", "tech", "capacity_mw", "source_table"]]
df_final = df_final.sort_values(["year", "tech"]).reset_index(drop=True)

assert df_final["year"].min() == 2010
assert df_final["year"].max() == 2024
assert not df_final.duplicated(["year", "tech"]).any()

os.makedirs("output", exist_ok=True)

output_path = "output/dukes_capacity_hist_2010_2024.parquet"
df_final.to_parquet(output_path, index=False)

print("DUKES 5.7 processing complete")
print("File:", output_path)
print("Rows:", len(df_final))
print("Duplicate year+tech rows:", df_final.duplicated(["year", "tech"]).sum())
print("Columns:", df_final.columns.tolist())
print("Years:", df_final.year.min(), "-", df_final.year.max())
print("Tech:", sorted(df_final["tech"].unique()))